In [ ]:
import pandas as pd
import geopandas as gpd
import altair as alt
import json
from shapely.geometry import shape, Point, LineString
from shapely import wkt
from shapely.ops import linemerge, unary_union
alt.data_transformers.disable_max_rows()
import h3
import geopandas as gpd
from shapely.ops import linemerge, substring, unary_union
from shapely.geometry import MultiLineString

In [105]:
nodes = pd.read_csv("Nodes/FGC.csv")
nodes['geometry']= nodes['geometry'].apply(wkt.loads)
nodes = gpd.GeoDataFrame(nodes, geometry='geometry', crs="EPSG:4326")

In [106]:
nodes

,id,name,linia,stop_type,geometry
0,F-L8-AL,Almeda,L8,FGC,POINT (2.08525 41.35307)
1,F-L8-LH,Av. Carrilet,L8,FGC,POINT (2.10271 41.35793)
2,F-L7-TB,Av. Tibidabo,L7,FGC,POINT (2.13716 41.40965)
3,F-S1-VL,Baixador de Vallvidrera,S1,FGC,POINT (2.09694 41.42009)
4,F-S1-PC,Catalunya,S1,FGC,POINT (2.16872 41.38563)
5,F-L7-PC,Catalunya,L7,FGC,POINT (2.16872 41.38563)
6,F-L6-PC,Catalunya,L6,FGC,POINT (2.16872 41.38563)
7,F-S1-PR,Diagonal,S1,FGC,POINT (2.15803 41.39281)
8,F-L7-PR,Diagonal,L7,FGC,POINT (2.15803 41.39281)
9,F-L6-PR,Diagonal,L6,FGC,POINT (2.15803 41.39281)


In [107]:
with open('data/FGC/fcg_trajectories.json') as f:
    data = json.load(f)
fgc_traj = []
for stop in data['features']:
    properties = stop.get('properties', {})
    route_id = properties.get('route_id')
    route_name = properties.get('route_long_name')

    
    fgc_traj.append({
        'route_id': route_id,
        'route_name': route_name,
        'geometry': shape(stop['geometry'])

    })  
geo_df_fgc_traj = gpd.GeoDataFrame(fgc_traj, crs="EPSG:4326")
valid = ['L6', 'L7', 'L8', 'L12','S1',]
geo_df_fgc_traj = geo_df_fgc_traj[geo_df_fgc_traj['route_id'].isin(valid)]
geo_df_fgc_traj[geo_df_fgc_traj['route_id'] == 'L6']

,route_id,route_name,geometry
0,L6,Barcelona Pl. Catalunya - Sarrià,"MULTILINESTRING ((2.17006 41.38575, 2.16973 41..."


In [108]:
line = geo_df_fgc_traj[geo_df_fgc_traj['route_id'] == 'L6'].geometry.iloc[0]
stops = nodes[nodes['linia'] == 'L6'].geometry.tolist()
stops

[<POINT (2.169 41.386)>,
 <POINT (2.158 41.393)>,
 <POINT (2.153 41.399)>,
 <POINT (2.136 41.398)>,
 <POINT (2.131 41.398)>,
 <POINT (2.142 41.399)>,
 <POINT (2.147 41.401)>,
 <POINT (2.126 41.398)>]

In [ ]:
import geopandas as gpd
from shapely.ops import linemerge, substring, unary_union
from shapely.geometry import MultiLineString


flattened_line = unary_union(line) 
merged_line = linemerge(flattened_line)


if merged_line.geom_type == 'MultiLineString':
    line_s = max(merged_line.geoms, key=lambda x: x.length)
else:
    line_s = merged_line

# 2. Project stops (standard procedure)
stop_list = []
for idx, row in nodes[nodes['linia'] == 'L6'].iterrows():
    dist = line_s.project(row.geometry)
    stop_list.append({'name': row['name'], 'id': row['id'], 'dist': dist})
sorted_stops = sorted(stop_list, key=lambda x: x['dist'])

# 4. Create segments
segments_data = []
for i in range(len(sorted_stops) - 1):
    origin = sorted_stops[i]
    destination = sorted_stops[i+1]
    
    # Check for zero length (stops at the same location)
    if abs(destination['dist'] - origin['dist']) < 1e-7:
        continue
        
    seg_geom = substring(line_s, origin['dist'], destination['dist'])
    
    segments_data.append({
        'origin': origin['id'],
        'destination': destination['id'],
        'tram': f"{origin['name']} - {destination['name']}",
        'geometry': seg_geom,
    })

segments_gdf = gpd.GeoDataFrame(segments_data, crs="EPSG:4326")

In [122]:
lines = alt.Chart(segments_gdf).mark_geoshape(
    filled=False, 
    strokeWidth=4
).encode(
    color=alt.Color('tram:N', legend=alt.Legend(title="Metro Segments")),
    tooltip=['tram:N']
)

points = alt.Chart(nodes[nodes['linia'] == 'L6']).mark_geoshape(size=0, color='red').encode(tooltip=['name:N'])
plot = (lines + points).project('mercator').properties(width=800, height=600)
plot
    

alt.LayerChart(...)